In [1]:
#!/usr/bin/env python
# coding: utf-8

import cv2
import numpy as np
from ipywidgets import widgets
from IPython.display import display
import time
import YB_Pcb_Car  # 假設你的類存成 YB_Pcb_Car.py

# -------初始化小車--------
car = YB_Pcb_Car.YB_Pcb_Car()
speed_base = 35  # 基本速度，可調整
speed_stop = 0
count = 0
turning = False
drive = True

# -------初始化鏡頭--------
car.Ctrl_Servo(1, 100)
time.sleep(0.5)
car.Ctrl_Servo(2, 110)
time.sleep(0.5)

# ---初始化攝影機與 widget---
image_widget = widgets.Image(format='jpeg', width=320, height=240)
display(image_widget)

def bgr8_to_jpeg(frame, quality=75):
    return bytes(cv2.imencode('.jpg', frame)[1])

cap = cv2.VideoCapture(0)
cap.set(3, 320)
cap.set(4, 240)
cap.set(5, 30)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M','J','P','G'))
cap.set(cv2.CAP_PROP_BRIGHTNESS, 60)
cap.set(cv2.CAP_PROP_CONTRAST, 50)
cap.set(cv2.CAP_PROP_EXPOSURE, 10)
cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.75) 
print(cap.get(cv2.CAP_PROP_AUTO_EXPOSURE))

# -------主迴圈-------
try:
    car.Car_Run(speed_base, speed_base)
    time.sleep(0.5)

    while drive:
        
        ret, frame = cap.read()
        if not ret:
            print("Camera read failed")
            break

        h, w, _ = frame.shape #(240,320)

        roi = frame
        

        # HSV
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

        # 黃色遮罩
        lower_yellow = (15, 50, 50)
        upper_yellow = (35, 255, 255)
        mask_yellow = cv2.inRange(hsv, lower_yellow, upper_yellow)

        # 白色遮罩
        lower_white = (15, 50, 50)
        upper_white = (35, 255, 255)
        mask_white = cv2.inRange(hsv, lower_white, upper_white)

        mask = cv2.bitwise_or(mask_yellow, mask_white)

        # 邊緣檢測
        edges = cv2.Canny(mask, 30, 100)

        # 霍夫直線
        lines = cv2.HoughLinesP(edges, 1, np.pi/180, 30, minLineLength=30, maxLineGap=60)

        x_left, x_right = [], []
        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                slope = (y2 - y1) / (x2 - x1 + 1e-6)
                if slope < -0.75:
                    x_left.extend([x1, x2])
                    cv2.line(roi, (x1, y1), (x2, y2),(0,255,0),2)
                elif slope > 0.75:
                    x_right.extend([x1, x2])
                    cv2.line(roi, (x1, y1), (x2, y2),(255,0,0),2)

        direction = "No lane detected"                
                
        if x_left:  # 只偵測到左線
            lx = np.mean(x_left)
            error = lx - (w // 8)  # 把左線視為目標，車身稍微偏右
            if error > 15:
                direction = "Right adjusting"
                car.Control_Car(speed_base * 7 // 4, speed_base) #向右微調
            else:
                direction = "Left adjusting"
                car.Control_Car(speed_base, speed_base * 12 // 4) #向左微調
            turning = False

        elif not x_left and not turning and count == 0:
            direction = "Stop"
            car.Car_Run(speed_stop, speed_stop)
            time.sleep(1.0)
            car.Car_Run(speed_base * 8 // 4, speed_base)
            time.sleep(0.4)
            direction = "Go straight"
            car.Car_Run(speed_base * 8 // 4, speed_base)
            time.sleep(0.8)
            turning = True
            
        elif not x_left and turning:
            direction = "Left turn"
            car.Car_Run(speed_base * 4 // 8, speed_base * 17 // 8)
            count = 1
            car.Ctrl_Servo(1, 90) #-
            car.Ctrl_Servo(2, 110) #-
            
        else:
            direction = "Arrived"
            car.Car_Run(speed_base, speed_base)
            time.sleep(0.6)
            car.Car_Stop()
            time.sleep(0.5)
            drive = False
        #print(np.mean(x_left),np.mean(x_right))
        # 更新影像 widget
        cv2.putText(roi, direction, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
        image_widget.value = bgr8_to_jpeg(roi)

        # ESC 停止
        if cv2.waitKey(1) & 0xFF == 27:
            break

except KeyboardInterrupt:
    print("Stopped by user")

finally:
    cap.release()
    cv2.destroyAllWindows()
    car.Car_Stop()


Image(value=b'', format='jpeg', height='240', width='320')

1.0


In [25]:
cap.release()
cv2.destroyAllWindows()
car.Car_Stop()